# Reinforcement Learning: learning to act from reward, not from labels

_Reinforcement Learning_

**An agent acts in an environment, sees a reward, and tries to maximize total reward over time — nobody tells it the right answer.**

Reinforcement Learning (RL) is learning how to act from reward. There is no teacher
       handing you correct answers. There is a world you can act in, and a number — the reward — that you
       are trying to make large over time.

       The whole subject is one loop. At each time step the agent looks at the current state,
       chooses an action, and the environment answers with a reward and a next state. The
       agent's goal is not the next reward — it is the return: the total (discounted) reward summed over
       the whole future.

---

This notebook is a practice scaffold for the **Reinforcement Learning: learning to act from reward, not from labels** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — gymnasium + numpy

We build the bare reinforcement-learning loop on **CartPole** with a purely random agent. The point is to establish a baseline: the standard `make` / `reset` / `step` API that every later lesson reuses, and the score a policy must beat once it actually starts learning.

### Step 1 — Create the environment

`gymnasium.make` hands us **CartPole-v1**, the classic balance-a-pole-on-a-cart task. The agent earns a reward of `+1` for every step the pole stays upright; the episode ends when the pole falls or 500 steps elapse. Inspecting the observation and action spaces tells us what the agent sees (a 4-number cart/pole state) and what it can do (push left or right).

In [ ]:
import gymnasium
import numpy as np

# CartPole-v1: balance a pole on a cart.
# Reward = +1 per step the pole stays up.
# Episode ends when the pole falls or 500 steps pass.
# This make / reset / step / done API is used by EVERY later lesson.
env = gymnasium.make("CartPole-v1")

# Box(4,): the continuous cart/pole state.
print("observation space:", env.observation_space)

# Discrete(2): push left or push right.
print("action space:     ", env.action_space)

### Step 2 — Run the minimal RL loop with a random agent

This is the entire RL interaction loop. Each episode we `reset` to a start state, then repeatedly sample a **random** action, `step` the environment, and accumulate the reward into the return `total_reward`. There is no learning here at all — we just want to see how a policy that ignores the state performs.

In [ ]:
n_episodes = 20
returns = []

for ep in range(n_episodes):
    # reset returns the initial observation S_0 and an info dict.
    state, info = env.reset(seed=ep)
    total_reward = 0.0          # this becomes the return G_0 for the episode
    done = False

    while not done:
        action = env.action_space.sample()      # RANDOM action A_t
        state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward                  # accumulate R_{t+1}
        done = terminated or truncated          # episode over?

    returns.append(total_reward)
    print(f"episode {ep:2d}: return = {total_reward:.0f}")

env.close()

### Step 3 — Summarise the baseline

Averaging the returns gives the score any learning method must beat. A random policy keeps CartPole up only about 20 steps, so the mean return is low and noisy. Q-learning, DQN, or policy gradient all have to clear this baseline to count as learning.

In [ ]:
mean_return = np.mean(returns)
std_return = np.std(returns)

print(f"\nrandom-agent mean return over {n_episodes} episodes: "
      f"{mean_return:.1f} +/- {std_return:.1f}")
# A random policy keeps CartPole up only ~20 steps. Any LEARNING method
# (Q-learning, DQN, policy gradient) must beat this baseline.

## Visualize the data & results

_Does a RANDOM agent improve over time? It should NOT — establishing the flat, noisy baseline that any learning method must beat._

### Step 1 — Build a tiny gridworld MDP from scratch

Here we define a 6x6 gridworld by hand in pure numpy — no gym needed. States `0..35` are laid out as a grid (`state = row*6 + col`), the bottom-right cell is the goal (reward `+10` and the episode ends), every other move costs `-1`, and bumping a wall leaves you in place. The `step` function encodes exactly these dynamics.

In [ ]:
import numpy as np

# A 6x6 gridworld MDP built from scratch in numpy (NO gym needed).
#   States 0..35 laid out as a 6x6 grid (state = row*6 + col).
#   Goal = bottom-right (state 35): reward +10 and the episode ENDS.
#   Every other step costs -1 (so dawdling hurts -> reaching the goal fast is good).
#   Actions: 0=up, 1=right, 2=down, 3=left. Bumping a wall keeps you put.
N_ROWS, N_COLS = 6, 6
GOAL = N_ROWS * N_COLS - 1            # state 35
MOVES = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}
STEP_COST, GOAL_REWARD = -1.0, 10.0

def step(s, a):
    r, c = divmod(s, N_COLS)
    dr, dc = MOVES[a]
    nr, nc = r + dr, c + dc
    if not (0 <= nr < N_ROWS and 0 <= nc < N_COLS):
        nr, nc = r, c                # bump into a wall -> stay
    ns = nr * N_COLS + nc
    if ns == GOAL:
        return ns, GOAL_REWARD, True
    return ns, STEP_COST, False

### Step 2 — Roll out the random agent for many episodes

The agent picks actions **uniformly at random** — no policy, no learning. We run 400 episodes (capped at 200 steps each), accumulating each episode's reward into its return. Because the agent never improves, the per-episode returns should stay poor and flat.

In [ ]:
rng = np.random.default_rng(0)
N_EPISODES, MAX_STEPS = 400, 200
returns = np.zeros(N_EPISODES)

for ep in range(N_EPISODES):
    s = 0                # start top-left (state 0)
    total = 0.0
    for _ in range(MAX_STEPS):
        a = rng.integers(4)          # RANDOM action: the baseline agent
        s, r, done = step(s, a)
        total += r                   # accumulate reward -> the return
        if done:
            break
    returns[ep] = total

print("mean return:", round(returns.mean(), 2))             # ~ -113 : poor, by design

# Compare the first 50 vs last 50 episodes: ~equal means NO learning.
first50 = round(returns[:50].mean(), 1)
last50 = round(returns[-50:].mean(), 1)
print("first-50 vs last-50 mean:", first50, last50)

### Step 3 — Smooth the curve into the flat baseline

A 20-episode running average smooths the noise so we can see the trend (or lack of one). We subsample it down to 40 points for a clean plot. The resulting line is flat — visual confirmation that random acting never improves, exactly the baseline a learning method must beat.

In [ ]:
# 20-episode running average -> the FLAT line we plot.
run = np.convolve(returns, np.ones(20) / 20, mode="valid")

# Subsample the running average down to 40 points.
idx = np.linspace(0, len(run) - 1, 40).astype(int)
points = [[int(i + 19), round(float(run[i]), 2)] for i in idx]
print("plotted points:", points)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** An episode yields rewards $R_1 = 2$, $R_2 = 0$, $R_3 = 0$, $R_4 = 5$ with discount $\gamma = 0.5$. Compute the return $G_0$ from the start.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the discounted sum: $G_0 = \gamma^0 R_1 + \gamma^1 R_2 + \gamma^2 R_3 + \gamma^3 R_4$. — _The return weights the reward $k$ steps ahead by $\gamma^k$; here $\gamma = 0.5$._
- Plug in: $1(2) + 0.5(0) + 0.25(0) + 0.125(5)$. — _$\gamma^0=1,\ \gamma^1=0.5,\ \gamma^2=0.25,\ \gamma^3=0.125$._
- Add: $2 + 0 + 0 + 0.625 = 2.625$. — _Only the non-zero rewards contribute._

**Answer:** $G_0 = 2.625$. The far-off $+5$ is discounted to $0.625$, so it counts for much less than the immediate $+2$.

</details>

**Problem 2.** Why can the undiscounted return ($\gamma = 1$) be a problem for a task that never ends, and how does $\gamma \lt 1$ fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- With $\gamma = 1$ every future reward counts fully, so an endless stream of positive rewards sums to infinity. — _You cannot compare two infinite returns — the objective becomes meaningless._
- With $0 \le \gamma \lt 1$ the weights $\gamma^k$ shrink geometrically. — _A bounded reward gives $|G_t| \le R_{\max}/(1-\gamma)$, a finite number, via the geometric series._

**Answer:** Discounting keeps the return finite (and well-defined) for never-ending tasks, while also encoding a preference for sooner rewards.

</details>

**Problem 3.** You have a dataset of labeled (image &rarr; correct caption) pairs. Should you use RL or supervised learning, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check whether a teacher provides the correct answer per input. Here, yes — each image has its correct caption. — _That is exactly the supervised setting: $(x, y)$ pairs with known $y$._
- RL would discard those labels and replace them with a noisy, delayed reward you must discover. — _RL is for when you can only SCORE behavior after the fact, not when the answer is handed to you._

**Answer:** Use supervised learning. It is far more sample-efficient and stable when correct labels exist; reserve RL for sequential, delayed-reward problems with no answer key.

</details>